# Etna Cause–Trigger Analysis


This notebook applies the Cause–Trigger workflow to the Etna case.

The main experiment evaluates

$$
5\ \text{minimum-}I_2\text{ values}
\times
12\ \text{maximum lag orders}
\times
3\ \text{causal-discovery backends}
=
180\ \text{configurations}.
$$

The backends are interpreted as follows:

- **HMML** is the baseline causal-discovery method used by the original
  Cause–Trigger implementation;
- **PCMCI** is a lagged causal-discovery extension;
- **PCMCI+** is an extension that additionally permits eligible directed
  contemporaneous links as trigger candidates.

Results are interpreted
after execution according to their recurrence across minimum-\(I_2\) values,
maximum lag orders, and backends. VAR-AIC and VAR-BIC are reported once as
conventional lag-order diagnostics but do not select or restrict the grid.

Two aligned representations of the same causal-analysis timestamps are used:

- `X_model`: case-standardised data for causal discovery, coefficient
  estimation, construction of \(V\), and moderation;
- `X_mean`: the same case observations standardised using the preceding
  reference interval, used to find the split and perform mean comparisons.

The reference observations are not included in \(I_1\) or \(I_2\). For each
minimum-\(I_2\) value, the split is found within `X_mean`; the same split index
then defines \(I_1\) and \(I_2\) in `X_model`.


Accepted pairs are displayed as **Cause | Trigger**. For Etna, the target
effect is anomalous local catalogue seismicity. The notebook displays compact grid summaries while
three CSV files preserve the complete run-level and moderation-level audit
trail.


## Imports and configuration


In [ ]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cause_trigger_cases import ETNA_CASE
from cause_trigger_reporting import (
    backend_min_i2_summary,
    delayed_lag_correlation,
    export_audit_csvs,
    pair_grid_stability_summary,
    plot_effect_with_splits,
    run_parameter_grid,
    show_table,
)
from cause_trigger_workflow import (
    COMPACT_RUN_SPECS,
    WorkflowConfig,
    case_study_interval,
    load_model_frame,
    pre_case_reference_interval,
    prepare_case_frames,
    reference_parameter_table,
    run_one,
    split_diagnostics,
)

CASE = ETNA_CASE
EFFECT = CASE.effect
MODEL_COLUMNS = CASE.model_columns
VARIABLE_LABELS = CASE.variable_labels

DATA_PATH = (
    PROJECT_ROOT / "data" / "etna" / "etna_dataset.csv"
)
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

SAVE_RESULTS = True
SHOW_FULL_GRID = False
SHOW_SCALING_REPORT = False

EVENT_TIME = pd.Timestamp("2008-05-12 06:28:00", tz="UTC")

CASE_PRE_DAYS = 3
CASE_POST_HOURS = 144

MIN_I1_LENGTH = 48
MIN_I2_GRID = (48, 60, 72, 84, 96)

MAX_LAGS = 12
LAG_GRID = tuple(range(1, MAX_LAGS + 1))

REFERENCE_DAYS = 14
REFERENCE_MIN_COVERAGE = 0.90

ALPHA = 0.05
PCMCI_PC_ALPHA = 0.05
PCMCI_ALPHA_LEVEL = 0.05
PCMCI_FDR_METHOD = "fdr_bh"
COND_IND_TEST = "robust_parcorr"

## Data, reference baseline, and grid-defined splits

The causal-analysis interval contains **3 days before** and
**144 hours after** the UTC hour containing the contextual event. The start and
end of this interval are fixed for every configuration.

The adjacent 14-day reference interval is used only to estimate transformation
offsets and reference-standardisation parameters. It does not enter the
causal-analysis interval.

The split search requires at least 48 h in \(I_1\). The minimum required length
of \(I_2\) is varied over `MIN_I2_GRID = (48, 60, 72, 84, 96)`. Because the
split depends on this value but not on the causal-discovery backend or maximum
lag order, the five split results are calculated and displayed once.


In [ ]:
X_full_raw = load_model_frame(
    DATA_PATH,
    include_columns=MODEL_COLUMNS,
    require_complete=False,
    index_name=CASE.index_name,
)

X_case_raw = case_study_interval(
    X_full_raw,
    EVENT_TIME,
    pre_days=CASE_PRE_DAYS,
    post_hours=CASE_POST_HOURS,
)

reference_raw = pre_case_reference_interval(
    X_full_raw,
    case_start=X_case_raw.index.min(),
    reference_days=REFERENCE_DAYS,
    min_coverage=REFERENCE_MIN_COVERAGE,
    case_name=CASE.name,
)

X_model, X_mean, scaling_report = prepare_case_frames(
    X_case_raw,
    reference_raw,
    case=CASE,
)

reference_parameters = reference_parameter_table(
    X_model,
    EFFECT,
    max_lags=MAX_LAGS,
    fallback_lag=1,
)

# This object is a template only. run_parameter_grid overwrites
# min_I2_length for every value in MIN_I2_GRID.
workflow_template = WorkflowConfig(
    effect=EFFECT,
    alpha=ALPHA,
    selected_lag=LAG_GRID[0],
    min_I1_length=MIN_I1_LENGTH,
    min_I2_length=MIN_I2_GRID[0],
    distribution="gaussian",
    refit_alpha=1.0,
    refit_cv=True,
    refit_cv_folds=3,
    pcmci_pc_alpha=PCMCI_PC_ALPHA,
    pcmci_alpha_level=PCMCI_ALPHA_LEVEL,
    pcmci_fdr_method=PCMCI_FDR_METHOD,
    pcmci_cond_ind_test=COND_IND_TEST,
    pcmci_plus_use_contemporaneous_triggers=False,
)

split_summary = pd.DataFrame([
    {
        "min_I2_length": int(min_i2_length),
        **split_diagnostics(
            X_mean,
            EFFECT,
            event_time=EVENT_TIME,
            min_I1_length=MIN_I1_LENGTH,
            min_I2_length=int(min_i2_length),
        ),
    }
    for min_i2_length in MIN_I2_GRID
])

split_display = split_summary[
    [
        "min_I2_length",
        "split_time",
        "I1_length",
        "I2_length",
        "split_score",
        "boundary_split",
        "distance_to_event",
    ]
].rename(columns={
    "min_I2_length": "Minimum I2 (h)",
    "split_time": "Split time",
    "I1_length": "I1 (h)",
    "I2_length": "I2 (h)",
    "split_score": "Split score",
    "boundary_split": "Boundary split",
    "distance_to_event": "Offset from event",
})

case_end_exclusive = X_case_raw.index.max() + pd.Timedelta("1h")
reference_start = reference_raw.attrs["reference_start"]
reference_end = reference_raw.attrs["reference_end"]
effect_peak = X_case_raw[CASE.raw_effect].idxmax()

design_summary = pd.DataFrame([
    {
        "Item": "Canonical dataset coverage",
        "Value": (
            f"{X_full_raw.index.min()} to {X_full_raw.index.max()} "
            f"({len(X_full_raw)} retained hourly rows)"
        ),
    },
    {
        "Item": "Causal-analysis interval",
        "Value": (
            f"[{X_case_raw.index.min()}, {case_end_exclusive}) "
            f"({len(X_case_raw)} h)"
        ),
    },
    {
        "Item": "Reference-standardisation interval",
        "Value": (
            f"[{reference_start}, {reference_end}); "
            f"{len(reference_raw)}/"
            f"{reference_raw.attrs['expected_rows']} h "
            f"({reference_raw.attrs['coverage']:.1%} coverage)"
        ),
    },
    {"Item": "Wenchuan earthquake origin time", "Value": str(EVENT_TIME)},
    {
        "Item": "Effect maximum",
        "Value": (
            f"{effect_peak} "
            f"({effect_peak - EVENT_TIME} from contextual event)"
        ),
    },
    {
        "Item": "Minimum interval design",
        "Value": (
            f"I1 >= {MIN_I1_LENGTH} h; "
            f"minimum I2 grid = {MIN_I2_GRID} h"
        ),
    },
    {
        "Item": "Maximum-lag-order design",
        "Value": (
            f"d = {LAG_GRID[0]}-{LAG_GRID[-1]} h for HMML, "
            "PCMCI, and PCMCI+"
        ),
    },
    {
        "Item": "Complete grid size",
        "Value": (
            f"{len(MIN_I2_GRID)} × {len(LAG_GRID)} × "
            f"{len(COMPACT_RUN_SPECS)} = "
            f"{len(MIN_I2_GRID) * len(LAG_GRID) * len(COMPACT_RUN_SPECS)} "
            "configurations"
        ),
    },
])

show_table(design_summary, "Prespecified design and timing")
show_table(split_display, "Effect split across minimum-I2 values")

if SHOW_SCALING_REPORT:
    show_table(
        scaling_report.reset_index(names="Variable"),
        "Transformation and scaling audit",
    )

plot_effect_with_splits(
    X_mean,
    EFFECT,
    split_summary,
    event_time=EVENT_TIME,
    event_label="Wenchuan earthquake",
    title=None,
    filename="etna_effect_splits",
    save_dir=FIGURES_DIR,
    formats=("pdf", "png"),
)

## Full minimum-\(I_2\) × backend × maximum-lag-order experiment

Every backend is run at every combination of

```python
MIN_I2_GRID = (48, 60, 72, 84, 96)
LAG_GRID = (1, 2, ..., 12)
```

A run with maximum lag order \(d=k\) permits lagged relations from 1 through
\(k\) hours; it does not test only lag \(k\).

The resulting 180 configurations constitute the main analysis. The grid is not
used to choose the smallest \(p\)-value. Interpretation is based on how accepted
pairs recur across minimum-\(I_2\) values, adjacent maximum lag orders, and
backends. HMML provides the baseline comparison, while PCMCI and PCMCI+ assess
how the findings change under the two alternative causal-discovery backends.


In [ ]:
experiment_grid, all_diagnostics = run_parameter_grid(
    X_model,
    X_mean,
    workflow_template,
    run_one=run_one,
    run_specs=COMPACT_RUN_SPECS,
    min_i2_values=MIN_I2_GRID,
    lags=LAG_GRID,
    cond_ind_test=COND_IND_TEST,
    metadata={
        "case_pre_days": CASE_PRE_DAYS,
        "case_post_hours": CASE_POST_HOURS,
        "reference_days": REFERENCE_DAYS,
    },
)

expected_grid_rows = (
    len(MIN_I2_GRID)
    * len(LAG_GRID)
    * len(COMPACT_RUN_SPECS)
)
if len(experiment_grid) != expected_grid_rows:
    raise RuntimeError(
        "Incomplete parameter grid: "
        f"expected {expected_grid_rows} rows, "
        f"received {len(experiment_grid)}."
    )

backend_summary = backend_min_i2_summary(
    experiment_grid,
    lag_count=len(LAG_GRID),
)

pair_summary = pair_grid_stability_summary(
    experiment_grid,
    min_i2_values=MIN_I2_GRID,
    lags=LAG_GRID,
    variable_labels=VARIABLE_LABELS,
)

show_table(
    backend_summary,
    "Backend results at each minimum-I2 setting",
)
show_table(
    pair_summary,
    "Accepted-pair support across the complete parameter grid",
)

grid_errors = experiment_grid.loc[
    experiment_grid["error"].notna()
].copy()

if not grid_errors.empty:
    show_table(
        grid_errors[
            [
                "min_I2_length",
                "run",
                "backend",
                "lag",
                "error",
            ]
        ],
        "Parameter-grid execution errors",
    )

if SHOW_FULL_GRID:
    show_table(
        experiment_grid.drop(columns="error"),
        "Complete minimum-I2 × backend × maximum-lag-order grid",
    )


## VAR information-criterion references

VAR-AIC and VAR-BIC are retained as conventional lag-order diagnostics. They
are shown once for comparison with the prespecified 1–12 h grid, but they do
not determine the grid and are not used to privilege any result.


In [ ]:
lag_reference_display = reference_parameters.rename(columns={
    "criterion": "Criterion",
    "selected_lag": "Selected maximum lag order (h)",
    "distribution": "Distribution",
    "max_lags": "Maximum order evaluated",
})

show_table(
    lag_reference_display,
    "Conventional VAR lag-order references",
)


## Etna-specific delayed-lag diagnostic

The catalogue-response episode may develop after the principal teleseismic
pulse. This separate descriptive scan evaluates the correlation between the
transformed teleseismic proxy at \(t-\tau\) and the effect at \(t\) over the
complete causal-analysis interval for delays up to 30 h.

This scan is not crossed with the minimum-\(I_2\) grid and is not treated as a
formal Cause–Trigger result.


In [ ]:
MAX_DELAY_HOURS = 30

delayed_lag_scan = delayed_lag_correlation(
    X_model,
    effect=EFFECT,
    predictor="teleseismic_scaled",
    max_lag=MAX_DELAY_HOURS,
)

top_delays = (
    delayed_lag_scan
    .assign(
        abs_correlation=lambda frame: frame["correlation"].abs()
    )
    .sort_values("abs_correlation", ascending=False)
    .drop(columns="abs_correlation")
    .head(10)
)

show_table(
    top_delays.rename(columns={
        "lag_hours": "Delay (h)",
        "correlation": "Correlation",
        "n_aligned": "Aligned rows",
    }),
    "Largest absolute full-case teleseismic-effect correlations",
)


## Saved audit outputs

The notebook writes three CSV files:

1. **All runs** — all 180 minimum-\(I_2\) × backend × maximum-lag-order
   configurations, including empty results, split information, parent sets,
   trigger candidates, accepted pairs, effective sample sizes, stop reasons,
   and errors.
2. **Moderation diagnostics** — every evaluated cause–trigger combination,
   accepted or rejected, with complete test statistics, reasons, and grid
   metadata.
3. **Analysis summary** — the prespecified design, conventional AIC/BIC
   references, all five split results, backend behaviour at each
   minimum-\(I_2\) value, and accepted-pair support across the complete grid.
   The Etna summary also includes the descriptive delayed-lag scan.


In [ ]:
if SAVE_RESULTS:
    saved_files = export_audit_csvs(
        results_dir=RESULTS_DIR,
        case_prefix="etna",
        experiment_grid=experiment_grid,
        diagnostics=all_diagnostics,
        design=design_summary,
        lag_references=reference_parameters,
        split_summary=split_summary,
        backend_summary=backend_summary,
        pair_summary=pair_summary,
        variable_labels=VARIABLE_LABELS,
        delayed_scan=delayed_lag_scan,
    )
    show_table(saved_files, "Saved result files")
